# Pipeline de Modelagem e Otimização de Hiperparâmetros

## Objetivos Principais
### Este notebook serve como um template padronizado e reutilizável para treinar, otimizar e comparar diferentes modelos de regressão multitarget. A partir do dataset processado, os objetivos são:

* #### **1. Treinamento Padronizado:** Aplicar um fluxo de trabalho consistente, carregando os dados limpos e utilizando a validação cruzada para avaliar a performance de diferentes algoritmos.
* #### **2. Otimização de Hiperparâmetros:** Utilizar `GridSearchCV` (ou `Optuna`) para encontrar a melhor combinação de hiperparâmetros para cada modelo testado, maximizando sua performance preditiva.
* #### **3. Comparação de Desempenho:** Registrar e comparar de forma sistemática os scores da validação cruzada entre os diferentes modelos (ex: Ridge vs. RandomForest vs. LGBM) para identificar a abordagem mais eficaz.
* #### **4. Flexibilidade para Experimentação:** Servir como uma base fácil de modificar. Para testar um novo algoritmo, basta alterar três pontos principais: o **import do modelo**, a **etapa do estimador no pipeline** e o **dicionário de hiperparâmetros**.

### Fluxo Lógico:

- **1. Separar o dataset em features e targets;**

- **2. Dividir as features e targets em conjunto de treino e teste;**

- **3. Escalonar as features:**
O objetivo do escalonamento é colocar todas as features na mesma ordem de grandeza. Isso é feito para evitar que algoritmos sensíveis à escala deem uma importância desproporcional a uma feature simplesmente porque seus valores numéricos são maiores. Esta etapa é CRÍTICA para modelos **lineares** como o `Ridge`, que são sensíveis à escala; só é **aplicado quando necessário**;

- **4. Transformar o problema de regressão multitarget** em problemas simples de prever cada target separadamente. 
Esta é a abordagem de _"problem transformation"_, que neste caso (com `MultiOutputRegressor`) não leva em conta a correlação entre os **alvos (targets);**também serão testadas diferentes abordagens, como _"RegressorChain"_ e _"Algorithm adaptation"_

- **5. Definir o regressor base:** Neste exemplo, a Regressão Ridge. 
É uma variação da Regressão Linear que combate o overfitting através de uma penalidade. O hiperparâmetro alpha (que controla a força da penalidade) não é definido neste exemplo, pois vários valores serão testados afim de encontrar o melhor;

- **6. Otimizar e Avaliar**:
O `GridSearchCV` ou `Optuna` realizam uma busca exaustiva (no primeiro caso) ou uma busca inteligente (no segundo caso) para encontrar os melhores hiperparâmetros (neste caso, o alpha) usando Validação Cruzada (`Cross Validation`) para garantir que a avaliação seja robusta e confiável;

- **7. Verificar os resultados e avaliar o modelo final:** 
Analisar os melhores hiperparâmetros encontrados pela busca e **verificar a performance do modelo otimizado** no conjunto de teste, que foi mantido separado durante todo o processo.

### Bibliotecas utilizadas:

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from joblib import dump

### Leitura dos dados processados:

In [8]:
df = pd.read_csv('../data/processed/proc_birds.csv')
df.head(3)

,normalize,feature preprocessing,mlfs.igmf_adapted.PyIT_IGMF,mlfs.igmf_adapted.PyIT_IGMF-n_features,mlfs.ppt_mi_adapted.PyIT_PPT_MI,mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features,mlfs.scls_adapted.PyIT_SCLS,mlfs.scls_adapted.PyIT_SCLS-n_features,mlfs.lrfs_adapted.PyIT_LRFS,mlfs.lrfs_adapted.PyIT_LRFS-n_features,...,weka.classifiers.functions.supportVector.NormalizedPolyKernel-E,weka.classifiers.functions.supportVector.NormalizedPolyKernel-L,weka.classifiers.functions.supportVector.Puk,weka.classifiers.functions.supportVector.Puk-O,weka.classifiers.functions.supportVector.Puk-S,weka.classifiers.functions.supportVector.RBFKernel,weka.classifiers.functions.supportVector.RBFKernel-G,F1 (macro averaged by label),Model Size,Model Size Log
0,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.256,37076.0,10.520752
1,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.222,29686.0,10.298465
2,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.026,18387.0,9.819454


### Lógica de treino do modelo e _hyperparameter tuning_

### Após obter o objeto do modelo treinado com os melhores hiperparâmetros e ter exibido suas métricas de avaliação, basta salvar essas métricas e o modelo treinado para reutilização futura

### Aqui termina o processo de treinamento e avaliação do modelo. O modelo utilizado aqui foi a Regressão Ridge, uma adaptação da regressão linear. Neste projeto ainda serão testados outros modelos, com outras abordagens e outras formas de otimização. Toda a implementação está disponível na pasta `/src`